# Image Color Palette Generator

Select an image and the number of colors to extract a palette. The generated palette shows each color's RGB and HEX values, along with its percentage of the image. Palettes can also be exported to CSV and individual HEX codes can be copied.

In [1]:
from pathlib import Path
from io import BytesIO
from html import escape
from PIL import Image
import base64

import ipywidgets as widgets
import pyperclip
import matplotlib.pyplot as plt
from IPython.display import display
from IPython.display import HTML

# Import the image analysis and visualization modules
from image_analyzer import ImageAnalyzer
from visualization import show_palette

## Palette Generation and Export Operations

In [2]:
# Keep track of the current palette and which colors are being displayed
current_palette = None
color_page = 0
colors_per_page = 5


def display_image(data, image_format):
    """Display the preview or generated palette."""
    # Convert image data so it can be displayed in an HTML widget
    encoded = base64.b64encode(data).decode("utf-8")

    if image_format == "jpeg":
        # Limit tall previews without changing landscape image sizes
        image_style = (
            "display:block;max-width:100%;max-height:400px;"
            "width:auto;height:auto;object-fit:contain;margin:0"
        )
    else:
        # Keep the generated visualization at its normal size
        image_style = (
            "display:block;width:100%;height:auto;margin:0"
        )

    image_display.value = f"""
    <img src="data:image/{image_format};base64,{encoded}"
         style="{image_style}">
    """


def update_preview():
    """Show the selected image before generating a palette."""
    try:
        with Image.open(image_select.value) as image:
            # Use a smaller copy for the preview without changing the original file
            preview = image.convert("RGB")
            preview.thumbnail((600, 400))

        buffer = BytesIO()
        preview.save(buffer, format="JPEG", quality=85)

        display_image(buffer.getvalue(), "jpeg")
        image_heading.value = "<b>Image preview</b>"

    except (OSError, ValueError) as error:
        image_display.value = f"Cannot preview image: {escape(str(error))}"


def copy_hex(hex_code):
    """Copy a HEX code to the clipboard."""
    try:
        pyperclip.copy(hex_code)
        status.value = f"<span style='color:#397A59'>Copied {hex_code}</span>"
    except pyperclip.PyperclipException as error:
        status.value = f"<span style='color:#B04A4A'>Copy failed: {escape(str(error))}</span>"


def update_color_list():
    """Show five clickable colors at a time."""
    if current_palette is None:
        return

    # Get the formatted colors and choose the current page
    records = current_palette.format_colors()
    start = color_page * colors_per_page
    visible_colors = records[start:start + colors_per_page]
    rows = []

    for record in visible_colors:
        hex_code = record["HEX"]

        # Create a small swatch showing the extracted color
        swatch = widgets.HTML(
            f"<div style='width:24px;height:24px;background:{hex_code};"
            f"border:1px solid #DDD;border-radius:5px'></div>",
            layout=widgets.Layout(width="28px")
        )

        # Clicking the HEX code copies it to the clipboard
        copy_button = widgets.Button(
            description=hex_code,
            tooltip=f"Copy {hex_code}",
            layout=widgets.Layout(width="105px", height="30px")
        )
        copy_button.add_class("palette-copy")
        copy_button.on_click(lambda _button, code=hex_code: copy_hex(code))

        percentage = widgets.HTML(
            f"<span style='font-size:12px;color:#666'>"
            f"{record['Percentage']:.1f}%</span>",
            layout=widgets.Layout(width="52px")
        )

        # Put the swatch, HEX button, and percentage on the same row
        rows.append(widgets.HBox(
            [swatch, copy_button, percentage],
            layout=widgets.Layout(
                width="225px",
                gap="5px",
                align_items="center"
            )
        ))

    color_list.children = tuple(rows)

    # Only show page controls if there are more than five colors
    total_pages = (len(records) + colors_per_page - 1) // colors_per_page
    color_navigation.layout.display = "flex" if total_pages > 1 else "none"

    # Update the page number and disable buttons when needed
    page_label.value = f"{color_page + 1} / {total_pages}"
    previous_button.disabled = color_page == 0
    next_button.disabled = color_page >= total_pages - 1


def previous_colors(_button):
    """Show the previous group of colors."""
    global color_page

    if color_page > 0:
        color_page -= 1
        update_color_list()


def next_colors(_button):
    """Show the next group of colors."""
    global color_page

    if current_palette is not None:
        total_pages = (len(current_palette) + colors_per_page - 1) // colors_per_page

        if color_page < total_pages - 1:
            color_page += 1
            update_color_list()


def generate_palette(_button):
    """Generate and display the color palette."""
    global current_palette, color_page

    # Clear the previous results and disable buttons during generation
    current_palette = None
    color_page = 0
    generate_button.disabled = True
    export_button.disabled = True
    color_section.layout.display = "none"
    status.value = "<span style='color:#777'>Generating palette...</span>"

    try:
        # Extract the colors using ImageAnalyzer
        analyzer = ImageAnalyzer(image_select.value)
        palette = analyzer.extract_dominant_colors(color_count.value)

        # Create the figure using the existing visualization module
        figure = show_palette(
            analyzer.image,
            palette,
            title=image_select.value.name,
            show=False
        )

        # Save the figure in memory so it can be shown in the interface
        buffer = BytesIO()
        try:
            figure.savefig(
                buffer,
                format="png",
                dpi=110,
                bbox_inches="tight",
                pad_inches=0.05
            )
        finally:
            plt.close(figure)

        # Replace the original preview with the generated palette
        display_image(buffer.getvalue(), "png")
        image_heading.value = "<b>Generated palette</b>"

        # Store the palette and enable exporting
        current_palette = palette
        update_color_list()
        color_section.layout.display = "flex"
        export_button.disabled = False

        status.value = (
            f"<span style='color:#397A59'>"
            f"Generated {len(palette)} colors successfully"
            f"</span>"
        )

    except Exception as error:
        # Restore the preview if generation fails
        current_palette = None
        update_preview()
        status.value = (
            f"<span style='color:#B04A4A'>"
            f"Could not generate palette: {escape(str(error))}"
            f"</span>"
        )

    finally:
        # Allow the user to try generating another palette
        generate_button.disabled = False


def export_palette(_button):
    """Save the current palette to CSV."""
    if current_palette is None:
        return

    try:
        # Name the CSV after the selected image
        filename = image_select.value.stem + "_palette.csv"
        saved_path = current_palette.export_palette(Path("output") / filename)

        status.value = (
            f"<span style='color:#397A59'>"
            f"Saved to {escape(str(saved_path))}"
            f"</span>"
        )

    except (ValueError, OSError) as error:
        status.value = (
            f"<span style='color:#B04A4A'>"
            f"Export failed: {escape(str(error))}"
            f"</span>"
        )


def reset_palette(_change):
    """Clear the palette when the settings change."""
    global current_palette, color_page

    # Prevent exporting an old palette after changing the inputs
    current_palette = None
    color_page = 0
    export_button.disabled = True
    color_list.children = ()
    color_section.layout.display = "none"
    status.value = "<span style='color:#777'>Ready to generate</span>"
    update_preview()


def on_image_change(change):
    """Update the image preview when another image is selected."""
    reset_palette(change)

## User Interface

In [3]:
# Find the JPG and PNG images available in the images folder
image_paths = sorted(
    path for path in Path("images").iterdir()
    if path.suffix.lower() in (".png", ".jpg", ".jpeg")
)

# Let the user select an image and choose the palette size
image_select = widgets.Dropdown(
    options=[(path.name, path) for path in image_paths],
    layout=widgets.Layout(width="240px")
)

color_count = widgets.BoundedIntText(
    value=5,
    min=1,
    max=256,
    layout=widgets.Layout(width="85px")
)

# Buttons for generating and exporting palettes
generate_button = widgets.Button(
    description="Generate Palette",
    icon="paint-brush",
    layout=widgets.Layout(width="240px", height="38px")
)
generate_button.style.button_color = "#DCCCF1"
generate_button.add_class("palette-generate")

export_button = widgets.Button(
    description="Export CSV",
    icon="download",
    disabled=True,
    layout=widgets.Layout(width="240px", height="38px")
)
export_button.style.button_color = "#DCE9E0"
export_button.add_class("palette-export")

# Display the original image or generated palette in the same area
image_display = widgets.HTML(
    layout=widgets.Layout(width="600px", margin="0")
)

image_heading = widgets.HTML("<b>Image preview</b>")
status = widgets.HTML("<span style='color:#777'>Ready to generate</span>")

# Store the clickable color entries without adding scrollbars
color_list = widgets.VBox(
    layout=widgets.Layout(
        width="225px",
        gap="3px",
        overflow="hidden"
    )
)

# Allow the user to browse longer palettes five colors at a time
previous_button = widgets.Button(
    description="Previous",
    icon="chevron-left",
    layout=widgets.Layout(width="88px", height="30px")
)

next_button = widgets.Button(
    description="Next",
    icon="chevron-right",
    layout=widgets.Layout(width="70px", height="30px")
)

page_label = widgets.HTML(
    layout=widgets.Layout(width="45px")
)

color_navigation = widgets.HBox(
    [previous_button, page_label, next_button],
    layout=widgets.Layout(
        width="225px",
        gap="6px",
        align_items="center",
        display="none"
    )
)

# Keep the color list hidden until a palette is generated
color_section = widgets.VBox([
    widgets.HTML("<b>Extracted colors</b>"),
    color_list,
    color_navigation
], layout=widgets.Layout(
    width="240px",
    gap="8px",
    display="none"
))

# Connect user interactions to the program functions
generate_button.on_click(generate_palette)
export_button.on_click(export_palette)
previous_button.on_click(previous_colors)
next_button.on_click(next_colors)

# Reset the palette if either setting changes
image_select.observe(on_image_change, names="value")
color_count.observe(reset_palette, names="value")


def label(text):
    """Create a label for an input field."""
    return widgets.HTML(
        f"<div style='font-size:12px;font-weight:600;"
        f"color:#68636D;margin-bottom:3px'>{text}</div>"
    )


# Add rounded corners and hover effects to the buttons
styles = widgets.HTML("""
<style>
.palette-generate,
.palette-export,
.palette-copy {
    border-radius: 9px !important;
}

.palette-generate:hover:not(:disabled),
.palette-export:hover:not(:disabled) {
    filter: brightness(0.96);
}

.palette-copy {
    border: none !important;
    background: #F4F2F6 !important;
    color: #41414A !important;
    font-size: 12px !important;
    text-align: left !important;
}

.palette-copy:hover {
    background: #EAE3F5 !important;
}
</style>
""")


# Main heading
header = widgets.HTML("""
<div style="padding-bottom:12px;border-bottom:1px solid #E8E5EB">
    <div style="font-size:30px;font-weight:600;color:#333039">
        Image Color Palette Generator
    </div>
</div>
""")


# Place the controls and extracted color information on the left
settings = widgets.VBox([
    label("SELECT IMAGE"),
    image_select,
    label("NUMBER OF COLORS"),
    color_count,
    generate_button,
    export_button,
    status,
    color_section
], layout=widgets.Layout(
    width="250px",
    min_width="0",
    gap="9px",
    padding="0",
    overflow="hidden",
    align_items="flex-start"
))


# Show the preview or generated palette on the right
image_panel = widgets.VBox([
    image_heading,
    image_display
], layout=widgets.Layout(
    width="600px",
    min_width="0",
    gap="6px",
    padding="0",
    align_items="flex-start"
))


# Arrange the settings and image side by side
workspace = widgets.GridBox(
    children=[settings, image_panel],
    layout=widgets.Layout(
        width="866px",
        grid_template_columns="250px 600px",
        grid_gap="16px",
        align_items="flex-start"
    )
)


# Combine everything into the main interface
interface = widgets.VBox([
    styles,
    header,
    workspace
], layout=widgets.Layout(
    width="890px",
    padding="12px",
    gap="12px",
    align_items="stretch"
))

# Keep the notebook output from stretching beyond the interface
display(HTML("""
<style>
.cell-output-ipywidget-background {
    width: fit-content !important;
    max-width: 100% !important;
}
</style>
"""))

# Show the first image before the user generates a palette
update_preview()
display(interface)